In [1]:
import bleach
import emoji
import nltk
import pandas as pd
import numpy as np
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import (
    Bidirectional,
    Concatenate,
    Dense,
    Embedding,
    Input,
    Lambda,
    LSTM,
    Reshape,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
df = pd.read_csv("D:\RPP\COMBINEDTWEETS\output.csv")
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into a string
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

df['preprocessed_text'] = df['text'].apply(preprocess_text)


[nltk_data] Downloading package punkt to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\My
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
df['preprocessed_text']

0       rt morningjewshow speaking jew comedy tonight ...
1       ageface recognition thingno reason platform ca...
2       upside moment think network news hasnt booked ...
3       youre going think aboutcreate experience need ...
4       watching thread fb possible future ux speaker ...
                              ...                        
9995    question change view business question change ...
9996    online video overtake broadcast tv google tv c...
9997    murdochus band publishing brother board ntttnt...
9998    seesaw launch premium paidfor online tv servic...
9999    weekend favs may twenty two weekend favs may t...
Name: preprocessed_text, Length: 10000, dtype: object

In [4]:
df['label'] = 0
df.loc[:4999, 'label'] = 0
df.loc[5000:9999, 'label'] = 1

In [5]:
df.columns

Index(['id', 'text', 'source', 'user_id', 'truncated', 'in_reply_to_status_id',
       'in_reply_to_user_id', 'in_reply_to_screen_name', 'retweeted_status_id',
       'geo', 'place', 'contributors', 'retweet_count', 'reply_count',
       'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive',
       'num_hashtags', 'num_urls', 'num_mentions', 'created_at', 'timestamp',
       'crawled_at', 'updated', 'preprocessed_text', 'label'],
      dtype='object')

In [6]:
df.label.value_counts

<bound method IndexOpsMixin.value_counts of 0       0
1       0
2       0
3       0
4       0
       ..
9995    1
9996    1
9997    1
9998    1
9999    1
Name: label, Length: 10000, dtype: int64>

In [7]:
df["preprocessed_text"].head(5)

0    rt morningjewshow speaking jew comedy tonight ...
1    ageface recognition thingno reason platform ca...
2    upside moment think network news hasnt booked ...
3    youre going think aboutcreate experience need ...
4    watching thread fb possible future ux speaker ...
Name: preprocessed_text, dtype: object

In [8]:
df["preprocessed_text"].dtype

dtype('O')

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['preprocessed_text'])
sequences = tokenizer.texts_to_sequences(df['preprocessed_text'])

# Calculate sequence lengths
sequence_lengths = [len(seq) for seq in sequences]

# Choose a percentile (e.g., 90th percentile)
percentile = 100
max_length = int(np.percentile(sequence_lengths, percentile))

print(f"Chosen max_length based on {percentile}th percentile: {max_length}")

Chosen max_length based on 100th percentile: 25


In [8]:
embedding_dim = None

# Assuming 'glove.twitter.27B.50d.txt' is the path to the GloVe embeddings file
with open('D:\\RPP\\WORD EMBEDING\\glove.twitter.27B.50d.txt', encoding='utf-8') as f:
    # Read the first line to extract the embedding dimension
    first_line = f.readline().strip().split()
    embedding_dim = len(first_line) - 1  # Subtract 1 to exclude the word itself

print("Embedding Dimension:", embedding_dim)


Embedding Dimension: 50


In [9]:
import numpy as np
import pandas as pd

# Load GloVe embeddings
glove_embeddings = {}  

with open('D:\\RPP\\WORD EMBEDING\\glove.twitter.27B.50d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype='float32')
        glove_embeddings[word] = vector

# Function to convert words into embeddings
def words_to_embeddings(words):
    vectors = []
    for word in words:
        if word in glove_embeddings:
            vectors.append(glove_embeddings[word])
    if vectors:
        return np.array(vectors, dtype='float32')  
    else:
        return np.zeros((1, len(next(iter(glove_embeddings.values())))), dtype='float32')  # Return zero vector if no embeddings found

# Define maximum sequence length
max_sequence_length = 25

# Convert words to embeddings and store them in a new column
df['word_embeddings'] = df['preprocessed_text'].apply(words_to_embeddings)


print(df['word_embeddings'])


0       [[0.11915, 0.4452, 0.38761, -0.4449, 0.1867, 0...
1       [[-0.201, 0.14106, 0.28319, 0.77383, -0.006815...
2       [[0.083004, 1.0053, 0.26507, -0.098562, -0.184...
3       [[-0.76372, 0.84481, -0.9094, 0.16341, -0.1135...
4       [[-0.15325, 1.2877, 0.13286, -0.03173, -0.6424...
                              ...                        
9995    [[0.76968, 0.20422, -0.041911, 0.044059, -0.06...
9996    [[0.60109, -0.14074, 0.37932, 0.27512, 0.02018...
9997    [[0.065259, 0.27438, -0.014515, -0.29043, -0.2...
9998    [[0.29535, -0.30249, -0.58106, -0.035171, -0.2...
9999    [[-0.15325, 1.2877, 0.13286, -0.03173, -0.6424...
Name: word_embeddings, Length: 10000, dtype: object


In [10]:
word_embeddings = df['word_embeddings'].values
max_sequence_length = 0
embedding_dimensionality = len(word_embeddings[0][0])  
for embedding_sequence in word_embeddings:
    sequence_length = len(embedding_sequence)
    if sequence_length > max_sequence_length:
        max_sequence_length = sequence_length
# Print the maximum sequence length and dimensionality of word embeddings
print("Sequence Length (T):", max_sequence_length)
print("Dimensionality of Word Embeddings (D):", embedding_dimensionality)

Sequence Length (T): 134
Dimensionality of Word Embeddings (D): 50


In [11]:
metadata_features = ['num_hashtags', 'num_urls', 'truncated', 'retweet_count', 'favorite_count',
                     'in_reply_to_status_id', 'in_reply_to_user_id']

# Fill NaN values with zeros and convert to int64
df.loc[:, metadata_features] = df[metadata_features].fillna(0).astype('int32')

# Normalize metadata features
scaler = StandardScaler()
df[metadata_features] = scaler.fit_transform(df[metadata_features])

In [12]:
print(df["word_embeddings"].shape)
print(df[metadata_features].shape)
print(df['label'].shape)

(10000,)
(10000, 7)
(10000,)


In [13]:
X = df['word_embeddings']
y = df['label']

In [14]:
df['word_embeddings'].dtype

dtype('O')

In [15]:
df['label'].dtype

dtype('int64')

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
# Print the shapes of the datasets
print("Training set:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nTesting set:")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nValidation set:")
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

Training set:
X_train shape: (6400,)
y_train shape: (6400,)

Testing set:
X_test shape: (2000,)
y_test shape: (2000,)

Validation set:
X_val shape: (1600,)
y_val shape: (1600,)


In [17]:
word_embeddings = df['word_embeddings'].values
max_sequence_length = 0
embedding_dimensionality = len(word_embeddings[0][0])  
for embedding_sequence in word_embeddings:
    sequence_length = len(embedding_sequence)
    if sequence_length > max_sequence_length:
        max_sequence_length = sequence_length
# Print the maximum sequence length and dimensionality of word embeddings
print("Sequence Length (T):", max_sequence_length)
print("Dimensionality of Word Embeddings (D):", embedding_dimensionality)

Sequence Length (T): 134
Dimensionality of Word Embeddings (D): 50


In [18]:
# Extract word embeddings from the 'word_embeddings' column
X_raw = df['word_embeddings'].values
# Define the sequence length (T) and dimensionality of word embeddings (D)
T = 134  # Sequence length
D = 50  # Dimensionality of word embeddings (assuming all word embeddings have the same dimensionality)
# Convert word embeddings into numpy array and pad/truncate sequences to ensure uniform length
X_padded = pad_sequences(X_raw, maxlen=T, dtype='float32', padding='post', truncating='post')

In [19]:
X_padded.shape

(10000, 134, 50)

In [20]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
X=X_padded
y = df['label']
# Define the LSTM model
lstm_model = Sequential()
lstm_model.add(layers.Input(shape=(134,50)))
lstm_model.add(layers.LSTM(64, return_sequences=True))
lstm_model.add(layers.Dropout(0.2))
lstm_model.add(layers.LSTM(64, return_sequences=True))
lstm_model.add(layers.Dropout(0.2))
lstm_model.add(layers.LSTM(64, return_sequences=True))
lstm_model.add(layers.Dropout(0.2))
lstm_model.add(layers.Flatten())

# Compile the model
lstm_model.compile(optimizer=Adam(learning_rate=0.0001), 
                   loss=BinaryCrossentropy(), 
                   metrics=['accuracy', AUC(name='auc')])
full_features=lstm_model.predict(X)
# Print model summary
lstm_model.summary()


313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 63ms/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 134, 64)        │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 134, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 134, 64)        │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 134, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 134, 64)        │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 134, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8576)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 95,488 (373.00 KB)

 Trainable params: 95,488 (373.00 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
full_features.shape

(10000, 8576)

In [22]:
import numpy as np
from sklearn.preprocessing import StandardScaler
X_train_metadata = df[metadata_features].fillna(0).astype('float64')
# Normalize metadata features
scaler = StandardScaler()
X_train_metadata_normalized = scaler.fit_transform(X_train_metadata)
X_train_metadata_normalized.shape

(10000, 7)

In [23]:
import pandas as pd
full_features_df = pd.DataFrame(full_features)
metadata_features_df = pd.DataFrame(X_train_metadata_normalized)
# Concatenate horizontally
combined_df = pd.concat([full_features_df, metadata_features_df], axis=1)
# Save to CSV
combined_df.to_csv('D:\RPP\COMBINEDTWEETS\combined_features2.csv', index=False)

In [24]:
combined_df.value_counts

<bound method DataFrame.value_counts of           0         1         2         3         4         5         6     \
0     0.001604 -0.006334  0.002771  0.004146  0.002765  0.000375 -0.003086   
1     0.002261 -0.008158  0.003151  0.001016  0.001416 -0.000642 -0.003442   
2     0.000751 -0.004626  0.001380  0.004818  0.004546 -0.001173 -0.002255   
3    -0.000754 -0.006183 -0.000024 -0.000724 -0.003096  0.001900  0.000493   
4    -0.000423 -0.003302 -0.000224  0.005331  0.003767 -0.001994 -0.004020   
...        ...       ...       ...       ...       ...       ...       ...   
9995 -0.002727 -0.007789  0.003720  0.001006  0.000825 -0.000123 -0.002566   
9996  0.000364 -0.007237  0.003323  0.001913  0.000419 -0.002010 -0.003454   
9997  0.000978 -0.006749  0.002539  0.003664  0.002654  0.000872 -0.002863   
9998  0.000229 -0.007282  0.002931  0.003360  0.002903  0.000014 -0.003735   
9999 -0.000423 -0.003302 -0.000224  0.005331  0.003767 -0.001994 -0.004020   

          7         8  

In [26]:
combined_df.shape

(10000, 8583)

In [31]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Concatenate, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Load the dataset
df = pd.read_csv("D:\RPP\COMBINEDTWEETS\output.csv")  # Replace "your_dataset.csv" with your dataset path
# Assuming your dataset contains 'text' column for tweet content and 'label' column for classification (0 for human, 1 for bot)

# Text preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    preprocessed_text = ' '.join(tokens)
    return preprocessed_text

df['preprocessed_text'] = df['text'].apply(preprocess_text)

# Tokenize text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['preprocessed_text'])
sequences = tokenizer.texts_to_sequences(df['preprocessed_text'])

# Pad sequences
max_sequence_length = 100  # Define the maximum sequence length
X = pad_sequences(sequences, maxlen=max_sequence_length, padding='post', truncating='post')

# Define metadata features (assuming you have metadata columns in your dataset)
metadata_features = ['num_hashtags', 'num_urls', 'retweet_count', 'favorite_count']  # Adjust as per your dataset

# Normalize metadata features
scaler = StandardScaler()
df[metadata_features] = scaler.fit_transform(df[metadata_features])
metadata = df[metadata_features].values

# Define input layers
tweet_text_input = Input(shape=(max_sequence_length,), name='tweet_text_input')
metadata_input = Input(shape=(len(metadata_features),), name='metadata_input')

# Define LSTMTC model for text processing
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 50  # Assuming you're using GloVe embeddings
tweet_embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(tweet_text_input)
lstm_output = LSTM(64)(tweet_embedding_layer)

# Combine text and metadata inputs
concatenated_input = Concatenate()([lstm_output, metadata_input])

# Define a fully connected layer for classification
dense_layer1 = Dense(64, activation='relu')(concatenated_input)
output = Dense(1, activation='sigmoid')(dense_layer1)

# Define the model
model = Model(inputs=[tweet_text_input, metadata_input], outputs=output)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [44]:
# Split the combined data into train and test sets for text and metadata separately
X_train_text, X_test_text, X_train_metadata, X_test_metadata, y_train, y_test = train_test_split(X, metadata, df['label'], test_size=0.2, random_state=42)

# Train the model
for epoch in range(10):
    # Create a new model instance for each epoch
    model = Model(inputs=[tweet_text_input, metadata_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Fit the model
    model.fit({'tweet_text_input': X_train_text, 'metadata_input': X_train_metadata}, y_train, epochs=1, batch_size=32)

    # Evaluate the model
    loss, accuracy = model.evaluate({'tweet_text_input': X_test_text, 'metadata_input': X_test_metadata}, y_test)
    print("Epoch:", epoch + 1, "Test Loss:", loss, "Test Accuracy:", accuracy)


250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 50ms/step - accuracy: 0.7390 - loss: 0.5598
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7708 - loss: 0.5224
Epoch: 1 Test Loss: 0.518787682056427 Test Accuracy: 0.7735000252723694
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.7940 - loss: 0.4900
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7782 - loss: 0.5150
Epoch: 2 Test Loss: 0.5125090479850769 Test Accuracy: 0.7774999737739563
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.7972 - loss: 0.4873
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7782 - loss: 0.5065
Epoch: 3 Test Loss: 0.5066373944282532 Test Accuracy: 0.7774999737739563
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.7979 - loss: 0.4782
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7798 - loss: 0.4983
Epoch: 4 Test Loss: 0.5003196597099304 Test Accuracy: 0.7789999842643738
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.7935 - loss: 0.4794
63/63 ━━━━━━━━━━━━━━━━━━━━ 1

In [45]:
# Assuming 'model' is your trained Keras model
model.save('trained_model.keras')